In [ ]:
import pandas as pd
import numpy as np

# 1. Recarregar e preparar os dados limpos
df = pd.read_csv('../dados/dataset_fullstack_900_smarttv_privacidade.csv')
df = df.drop_duplicates(subset=['id_registro'])

# Padronizações textuais
df['categoria_conteudo'] = df['categoria_conteudo'].str.strip().replace({'FILMES': 'Filmes', 'séries': 'Series', 'ESPOrtes': 'Esportes', 'noticias': 'Noticias', 'infantil': 'Infantil'})
df['estado_microfone'] = df['estado_microfone'].str.strip().replace({'ativo': 'Ativo', 'INATIVO': 'Inativo'})
df['categoria_telemetria'] = df['categoria_telemetria'].str.strip().replace({'acr/conteudo': 'ACR/conteudo', 'publicidade': 'Publicidade', 'TELEMETRIA SISTEMA': 'Telemetria sistema', 'Metadados de Eventos': 'Metadados de eventos'})
df['consentimento_coleta'] = df['consentimento_coleta'].fillna('Indeterminado')

# Limpeza de inconsistências numéricas (viram NaN)
df.loc[(df['minutos_visualizados'] < 0) | (df['minutos_visualizados'] > 10), 'minutos_visualizados'] = np.nan
df.loc[(df['confianca_classificacao'] < 0) | (df['confianca_classificacao'] > 1), 'confianca_classificacao'] = np.nan
df.loc[df['trafego_saida_kb'] < 0, 'trafego_saida_kb'] = np.nan
df.loc[df['anuncios_exibidos'] < 0, 'anuncios_exibidos'] = np.nan

mask_int = df['interacoes_anuncios'] > df['anuncios_exibidos']
df.loc[mask_int, ['interacoes_anuncios', 'anuncios_exibidos']] = np.nan

# Flag para a ausência de confiança
mask_legitimo = (df['estado_microfone'] == 'Inativo') | (df['classe_som'] == 'Silencio')
df['ausencia_confianca_legitima'] = df['confianca_classificacao'].isna() & mask_legitimo

# Outliers de tráfego usando IQR
Q1 = df['trafego_saida_kb'].quantile(0.25)
Q3 = df['trafego_saida_kb'].quantile(0.75)
IQR = Q3 - Q1
limite_sup = Q3 + 1.5 * IQR
df['is_outlier_trafego'] = df['trafego_saida_kb'] > limite_sup


# 2. CÓDIGOS PARA AS INTERPRETAÇÕES 


print("=== VARIÂNCIA POPULACIONAL VS AMOSTRAL ===")
var_populacional = df['trafego_saida_kb'].var(ddof=0)
var_amostral = df['trafego_saida_kb'].var(ddof=1)
print(f"População (ddof=0): {var_populacional:.2f}")
print(f"Amostral  (ddof=1): {var_amostral:.2f}\n")

print("=== MODA DE VARIÁVEL CONTÍNUA ===")
moda_trafego = df['trafego_saida_kb'].mode()
print(f"Moda do tráfego (retorna uma Series vazia ou um valor aleatório isolado):\n{moda_trafego}\n")

print("=== LIMITE INFERIOR NEGATIVO ===")
limite_inf = Q1 - 1.5 * IQR
print(f"Limite Inferior: {limite_inf:.2f} KB\n")

print("=== ESTATÍSTICAS E ASSIMETRIA ===")
vars_cont = ['minutos_visualizados', 'confianca_classificacao', 'trafego_saida_kb', 'anuncios_exibidos']
for v in vars_cont:
    n_valid = df[v].count() # O count ignora NaNs automaticamente
    mean = df[v].mean()
    median = df[v].median()
    std = df[v].std(ddof=1) # Desvio-padrão amostral
    cv = (std / mean) * 100 if mean != 0 else np.nan
    print(f"[{v}] N Válido={n_valid} | Média={mean:.2f} | Mediana={median:.2f} | Std={std:.2f} | CV={cv:.2f}%")

print("\n=== OUTLIERS CONCENTRADOS (CROSSTAB) ===")
print(pd.crosstab(df['is_outlier_trafego'], df['modo_privacidade']))

print("\n=== COMPARAÇÕES POR GRUPO (GROUPBY) ===")
print("Estatísticas do Tráfego por Modo de Privacidade:")
print(df.groupby('modo_privacidade')['trafego_saida_kb'].describe())

print("\nEstatísticas do Tráfego por Consentimento de Coleta:")
print(df.groupby('consentimento_coleta')['trafego_saida_kb'].describe())

print("\n=== FREQUÊNCIA DE CATEGORIAS DE CONTEÚDO (normalize=True) ===")
freq_conteudo = df['categoria_conteudo'].value_counts(normalize=True) * 100
print(freq_conteudo.round(2))

=== VARIÂNCIA POPULACIONAL VS AMOSTRAL ===
População (ddof=0): 329698.83
Amostral  (ddof=1): 330083.10

=== MODA DE VARIÁVEL CONTÍNUA ===
Moda do tráfego (retorna uma Series vazia ou um valor aleatório isolado):
0    0.0
Name: trafego_saida_kb, dtype: float64

=== LIMITE INFERIOR NEGATIVO ===
Limite Inferior: -27.63 KB

=== ESTATÍSTICAS E ASSIMETRIA ===
[minutos_visualizados] N Válido=852 | Média=6.95 | Mediana=7.12 | Std=2.13 | CV=30.63%
[confianca_classificacao] N Válido=256 | Média=0.78 | Mediana=0.78 | Std=0.13 | CV=16.53%
[trafego_saida_kb] N Válido=859 | Média=76.53 | Mediana=23.56 | Std=574.53 | CV=750.73%
[anuncios_exibidos] N Válido=871 | Média=1.84 | Mediana=2.00 | Std=1.15 | CV=62.34%

=== OUTLIERS CONCENTRADOS (CROSSTAB) ===
modo_privacidade    Ativado  Desativado
is_outlier_trafego                     
False                   329         519
True                     11          23

=== COMPARAÇÕES POR GRUPO (GROUPBY) ===
Estatísticas do Tráfego por Modo de Privacidade:
   

In [3]:
# versão com função

import pandas as pd
import numpy as np

# 1. Preparação e Limpeza Rápida dos Dados (Etapas anteriores)
df = pd.read_csv('../dados/dataset_fullstack_900_smarttv_privacidade.csv')
df = df.drop_duplicates(subset=['id_registro'])

# Invalidar erros sistêmicos (passar para NaN)
df.loc[(df['minutos_visualizados'] < 0) | (df['minutos_visualizados'] > 10), 'minutos_visualizados'] = np.nan
df.loc[(df['confianca_classificacao'] < 0) | (df['confianca_classificacao'] > 1), 'confianca_classificacao'] = np.nan
df.loc[df['trafego_saida_kb'] < 0, 'trafego_saida_kb'] = np.nan
df.loc[df['anuncios_exibidos'] < 0, 'anuncios_exibidos'] = np.nan

mask_int = df['interacoes_anuncios'] > df['anuncios_exibidos']
df.loc[mask_int, ['interacoes_anuncios', 'anuncios_exibidos']] = np.nan

# ==========================================
# 2. FUNÇÃO ESTATÍSTICA REUTILIZÁVEL
# ==========================================
def calcular_estatisticas(df, coluna):
    # O dropna() garante que só vamos usar os dados válidos para a matemática
    serie = df[coluna].dropna()
    n = serie.count()
    
    # 10 Medidas Básicas
    media = serie.mean()
    mediana = serie.median()
    std = serie.std(ddof=1) # ddof=1 (amostral)
    cv = (std / media * 100) if media != 0 else np.nan
    
    minimo = serie.min()
    q1 = serie.quantile(0.25)
    q3 = serie.quantile(0.75)
    maximo = serie.max()
    
    # Limites e Outliers
    iqr = q3 - q1
    limite_inf = q1 - 1.5 * iqr
    limite_sup = q3 + 1.5 * iqr
    
    outliers = serie[(serie < limite_inf) | (serie > limite_sup)].count()
    
    # Retorna o dicionário com arredondamentos para 2 casas
    return {
        'Variável': coluna,
        'N Válido': n,
        'Média': round(media, 2),
        'Mediana': round(mediana, 2),
        'Desvio Padrão': round(std, 2),
        'CV (%)': round(cv, 2),
        'Mínimo': round(minimo, 2),
        'Q1': round(q1, 2),
        'Q3': round(q3, 2),
        'Máximo': round(maximo, 2),
        'IQR': round(iqr, 2),
        'Lim. Inf.': round(limite_inf, 2),
        'Lim. Sup.': round(limite_sup, 2),
        'Qtd Outliers': outliers
    }

# ==========================================
# 3. APLICAÇÃO E TABELA CONSOLIDADA
# ==========================================
colunas_interesse = [
    'minutos_visualizados', 
    'trafego_saida_kb', 
    'confianca_classificacao', 
    'anuncios_exibidos', 
    'interacoes_anuncios'
]

# Roda a função para cada coluna e junta tudo numa lista de dicionários
resultados = [calcular_estatisticas(df, col) for col in colunas_interesse]

# Transforma a lista de dicionários no DataFrame final
df_estatisticas = pd.DataFrame(resultados).set_index('Variável')

print(df_estatisticas.to_string())

                         N Válido  Média  Mediana  Desvio Padrão  CV (%)  Mínimo     Q1     Q3   Máximo    IQR  Lim. Inf.  Lim. Sup.  Qtd Outliers
Variável                                                                                                                                          
minutos_visualizados          852   6.95     7.12           2.13   30.63    0.00   5.51   8.63    10.00   3.12       0.82      13.32             1
trafego_saida_kb              859  76.53    23.56         574.53  750.73    0.00  12.18  38.72  9200.00  26.54     -27.63      78.54            34
confianca_classificacao       256   0.78     0.78           0.13   16.53    0.56   0.68   0.89     0.99   0.21       0.37       1.20             0
anuncios_exibidos             871   1.84     2.00           1.15   62.34    0.00   1.00   3.00     4.00   2.00      -2.00       6.00             0
interacoes_anuncios           876   0.26     0.00           0.54  209.53    0.00   0.00   0.00     3.00   0.00       0

In [6]:
import pandas as pd
import numpy as np

# Preparação e Limpeza
df = pd.read_csv('../dados/dataset_fullstack_900_smarttv_privacidade.csv')
df = df.drop_duplicates(subset=['id_registro'])

df.loc[(df['minutos_visualizados'] < 0) | (df['minutos_visualizados'] > 10), 'minutos_visualizados'] = np.nan
df.loc[(df['confianca_classificacao'] < 0) | (df['confianca_classificacao'] > 1), 'confianca_classificacao'] = np.nan
df.loc[df['trafego_saida_kb'] < 0, 'trafego_saida_kb'] = np.nan
df.loc[df['anuncios_exibidos'] < 0, 'anuncios_exibidos'] = np.nan

mask_int = df['interacoes_anuncios'] > df['anuncios_exibidos']
df.loc[mask_int, ['interacoes_anuncios', 'anuncios_exibidos']] = np.nan

# 1. Função Atualizada
def calcular_estatisticas(df, coluna):
    serie = df[coluna].dropna()
    n = serie.count()
    
    media = serie.mean()
    mediana = serie.median()
    
    # Moda (tratando para pegar a primeira em caso de empate)
    moda_series = serie.mode()
    moda = moda_series.iloc[0] if not moda_series.empty else np.nan
    
    # Variância e Desvio-Padrão Amostrais (ddof=1)
    var = serie.var(ddof=1)
    std = serie.std(ddof=1)
    cv = (std / media * 100) if media != 0 else np.nan
    
    minimo = serie.min()
    maximo = serie.max()
    amplitude = maximo - minimo
    
    q1 = serie.quantile(0.25)
    q3 = serie.quantile(0.75)
    iqr = q3 - q1
    
    limite_inf = q1 - 1.5 * iqr
    limite_sup = q3 + 1.5 * iqr
    
    outliers = serie[(serie < limite_inf) | (serie > limite_sup)].count()
    
    return {
        'Variável': coluna,
        'N Válido': n,
        'Média': round(media, 2),
        'Mediana': round(mediana, 2),
        'Moda': round(moda, 2) if pd.notnull(moda) else np.nan,
        'Variância': round(var, 2),
        'Desvio Padrão': round(std, 2),
        'CV (%)': round(cv, 2),
        'Amplitude': round(amplitude, 2),
        'Mínimo': round(minimo, 2),
        'Q1': round(q1, 2),
        'Q3': round(q3, 2),
        'Máximo': round(maximo, 2),
        'IQR': round(iqr, 2),
        'Lim. Inf.': round(limite_inf, 2),
        'Lim. Sup.': round(limite_sup, 2),
        'Qtd Outliers': outliers
    }

colunas = ['minutos_visualizados', 'trafego_saida_kb', 'confianca_classificacao', 'anuncios_exibidos', 'interacoes_anuncios']
df_estatisticas = pd.DataFrame([calcular_estatisticas(df, c) for c in colunas]).set_index('Variável')
print(df_estatisticas.to_string())

                         N Válido  Média  Mediana   Moda  Variância  Desvio Padrão  CV (%)  Amplitude  Mínimo     Q1     Q3   Máximo    IQR  Lim. Inf.  Lim. Sup.  Qtd Outliers
Variável                                                                                                                                                                       
minutos_visualizados          852   6.95     7.12  10.00       4.54           2.13   30.63      10.00    0.00   5.51   8.63    10.00   3.12       0.82      13.32             1
trafego_saida_kb              859  76.53    23.56   0.00  330083.10         574.53  750.73    9200.00    0.00  12.18  38.72  9200.00  26.54     -27.63      78.54            34
confianca_classificacao       256   0.78     0.78   0.59       0.02           0.13   16.53       0.43    0.56   0.68   0.89     0.99   0.21       0.37       1.20             0
anuncios_exibidos             871   1.84     2.00   2.00       1.31           1.15   62.34       4.00    0.00   1.00   3

Os cálculos estatísticos foram realizados usando ddof=1, tratando os dados como uma amostra (Variância Amostral). A justificativa técnica é que este dataset representa um retrato estático de 882 registros de interações coletadas em um momento específico do tempo. Não possuímos os logs de todos os aparelhos já ativados, tampouco cobrimos toda a vida útil do ecossistema destas Smart TVs. Ao usarmos o ddof=1 (Correção de Bessel), evitamos subestimar a verdadeira variância do conjunto populacional do qual essas 882 medições foram retiradas.

Interpretação das Variáveis Contínuas
Tráfego de Saída: É a variável mais discrepante. Possui uma amplitude massiva de 9.200 KB e uma variância assombrosa (330.083,10). A enorme distância entre a média (76,53 KB) e a mediana (23,56 KB) reflete uma intensa assimetria à direita. A maior parte dos aparelhos opera de maneira leve, mas um grupo seleto (onde os 34 outliers residem) puxa toda a média para cima com envios severos de dados.

Minutos Visualizados: A distribuição de visualizações por registro tem uma amplitude máxima fechada (10,00 minutos). A simetria dos dados é notável, com a média (6,95) e a mediana (7,12) confortavelmente alinhadas e um coeficiente de variação bem contido (30,63%). Interessante observar que a moda cravou em 10,00, indicando que muitas sessões atingem o tempo limite exato de cobertura do pacote coletado.

Confiança de Classificação: É o indicador mais estável tecnicamente. Para os 256 momentos válidos, a média e a mediana convergem em 0,78, atestando forte simetria. A amplitude é estreita (0,43) e não apresenta sequer um outlier estatístico, provando que quando o algoritmo de inteligência consegue classificar um som e atribuir uma nota, o comportamento do modelo é previsível.

Anúncios e Interações: A exibição de anúncios é padronizada e estável (moda e mediana cravadas em 2,00). Por outro lado, a coluna de interações demonstra que o clique nos anúncios é raro e isolado: a mediana e o Q3 estão fixos em 0,00. Isso faz com que todo clique bem-sucedido figure fora do limite de densidade, o que sinaliza os 187 registros interativos matematicamente como outliers no uso da base geral.